# पाठ 18 (अनुवर्ती): रसीदें जो प्रमाणित करती हैं कि एक *मानव* ने कार्रवाई को अधिकृत किया

यह पाठ साबित करता है कि **एजेंट** ने क्या किया और **गेट** ने क्या निर्णय लिया। यह नोटबुक वह खोया हुआ आधा जोड़ता है: प्रमाण कि एक **नामित मानव** ने **ठीक वही** कार्रवाई को मंजूरी दी — एक अलग, मानव-धारित हस्ताक्षर पूरे कैनोनिकल कार्रवाई पर, ऑफ़लाइन सत्यापित।

यहां दोनों कलाकृतियां उसी **लीसन की रसीदों के समान लिफाफे के आकार** का उपयोग करती हैं: एक समतल payload जिसमें एक `type` फ़ील्ड है, जो कैनोनिकल JCS बाइट्स पर सीधे Ed25519 से हस्ताक्षरित है, जिसमें एक संरचित `signature` ऑब्जेक्ट जुड़ा है (और हस्ताक्षरित बाइट्स से बाहर रखा गया)। अनुमोदन रसीद एक नया `type` (`human.approval.v1`) है जो कार्रवाई प्रकार के साथ है, इसलिए एक `verify_chain` दोनों कलाकृतियों को उसी कोड पथ के साथ कवर करता है जो आपने मुख्य नोटबुक में बनाया। यह मानव-स्वीकृति रसीद एक शैक्षिक संरचना है जो यहां परिभाषित की गई है, न कि draft-farley-acta-signed-receipts द्वारा परिभाषित कोई रसीद प्रकार।

मुख्य नोटबुक में डेमो सत्यापक पर एक जानबूझ कर उन्नयन: यहां सत्यापक `signature.key_id` को रसीद के अंदर ले जाए गए सार्वजनिक कुंजी पर भरोसा करने के बजाय **पिन की गई कुंजी रजिस्ट्री** के विरुद्ध हल करता है। यही उत्पादन की अवस्था है जिसे पाठ की अपनी चेकलिस्ट सुझाती है ("सत्यापन सार्वजनिक कुंजी प्रकाशित करें"), और यही वह है जो जालसाजी को अस्वीकृति बनाता है बजाय अपनी-कुंजी-लीजिए बाईपास के।

इस नोटबुक का नियम: **हस्ताक्षरित अनुमोदन स्वयं में अधिकार नहीं है।** अधिकार तभी होता है जब अनुमोदन रसीद और क्रिया रसीद अभी भी निष्पादन समय पर उसी कैनोनिकल क्रिया से बंधी हो, एक नीति संस्करण, कुंजी, और समाप्ति के अंतर्गत जो अभी भी वर्तमान हों, और ऐसी अनुमोदन जो पहले से उपयोग नहीं हुई हो। हर असफलता एक **अलग कारण** के साथ अस्वीकृत होती है, ताकि आप *अधिकार समय-सीमा समाप्त हो गया* और *कार्यवाई बदल गई* को अलग-अलग पहचान सकें।


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## सटीक कार्रवाई

अनुमोदन की इकाई **कैनेनिकल एक्शन ऑब्जेक्ट** है — "रिफंड अनुमोदित करें" जैसे अस्पष्ट लेबल नहीं, बल्कि सटीक, पूर्ण रूप से निर्दिष्ट कार्रवाई। पूरे ऑब्जेक्ट पर हस्ताक्षर करना (और उससे एक डाइजेस्ट निकालना) हमें बाद में यह प्रमाणित करने की अनुमति देता है कि मानवीय ने *यह* और कुछ नहीं अनुमोदित किया।


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## एक लिफाफा, दो अधिकारियाँ

हर रसीद सबक का लिफाफा होती है: एक सपाट पेलोड जिसमें `type` फ़ील्ड होती है, साथ ही एक `signature` ऑब्जेक्ट (`alg`, `sig`, `key_id`) जो कि **साइन किए गए बाइट्स का हिस्सा नहीं** होता। `verify_envelope` दोनों प्रकार की रसीदों के लिए साझा संरचनात्मक + सिग्नेचर जांच है; जिस **पिन की रजिस्ट्री** पर यह `signature.key_id` को हल करता है, वही अधिकारियाँ अलग रखती है:

- **स्वीकृति रसीद** (`human.approval.v1`) — नामित अनुमोदक, पूरा कैनोनिकल कार्य **और उसका डाइजेस्ट**, `policy_version`, जारी करने और समाप्ति के टाइमस्टैंप। एक बार उपभोग करना चेन स्तर पर ट्रैक किया जाता है।
- **कार्य रसीद** (`agent.action.v1`) — एजेंट पहचान, `run_id`, वही कैनोनिकल कार्य **डाइजेस्ट**, निष्पादन परिणाम + टाइमस्टैंप, और `parent_approval_ref`: अनुमोदन का `receipt_hash`, सबक की चेन में `previous_receipt_hash` जैसी ही कन्वेंशन।

साझा `action_digest` फ़ील्ड वह जुड़ाव है जिस पर बाइंडिंग निर्भर करती है। `key_id` केवल एक लुकअप संकेत के रूप में सिग्नेचर ऑब्जेक्ट में रहता है: इसे एक अलग पिन की की ओर इशारा करने से सिग्नेचर जांच विफल हो जाती है, इसलिए यह कुछ भी प्रदान नहीं करता।


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: जहां बाइंडिंग वास्तव में तय होती है

`verify_chain` दो हस्ताक्षर जांचों के ऊपर एक सुविधा रैपर **नहीं** है। यह एकमात्र स्थान है जहाँ साझा कानूनी `action_digest`, अनुमोदन की नीति/कुंजी/समाप्ति की **ताजगी**, और अनुमोदन के **एक बार उपयोग** को साथ में जांचा जाता है, उस क्रिया के खिलाफ जो *अभी अभी* निष्पादित की जा रही है।

प्रत्येक अस्वीकृति एक **भिन्न कारण** के साथ इंकार करती है, ताकि अस्वीकृति पढ़ने वाला यह बता सके कि अधिकार कब पुराना हो गया (नीति बदल गई, कुंजी घूमाई गई, अनुमोदन समाप्त हो गया, अनुमोदन उपयोग हो गया) या निष्पादित क्रिया अभी भी वैध अनुमोदन के तहत बदल गई (डाइजेस्ट प्रतिस्थापन)।


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## बाइंडिंग क्या पकड़ता है

नीचे हर मामला **अलग कारण** के साथ **बंद** हो जाता है। पहला ब्लॉक क्लासिक सेट है (टैम्पर, भ्रमित डिप्टी, रप्ले, किसी भी प्राधिकरण पर फॉर्जरी, गलत इनपुट)। दूसरा ब्लॉक वह जो संपत्ति को सच करता है बजाय केवल घोषित करने के:

- **पुराना प्राधिकरण** — हस्ताक्षर अभी भी वैध है, लेकिन नीति संस्करण बदल गया है, अनुमोदक कुंजी पिन्ड रजिस्ट्री से हटा दी गई है, या अनुमोदन निष्पादन से पहले समाप्त हो गया;
- **डाइजेस्ट प्रतिस्थापन** — एक वैध रूप से हस्ताक्षरित क्रिया रसीद जिसकी `parent_approval_ref` एक *वास्तविक* अनुमोदन की ओर इशारा करती है, लेकिन उस अनुमोदन काCanonical क्रिया डाइजेस्ट उस क्रिया से मेल नहीं खाता जो वास्तव में निष्पादित हो रही है।


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## यह क्या साबित करता है — और क्या नहीं

**साबित करता है:** एक नामित मानव ने *इस सटीक कैनॉनिकल क्रिया* (पूर्ण क्रिया + डाइजेस्ट, पिन किए गए रजिस्ट्री से हल की गई कुंजी के साथ हस्ताक्षरित) को मंजूरी दी, और एजेंट ने *सटीक वही मंजूर क्रिया* (एक ही डाइजेस्ट, `receipt_hash` द्वारा मंजूरी से बंधा हुआ रसीद, पाठ का अपना चेन प्रथा) निष्पादित की — जबकि मंजूरी की नीति संस्करण, कुंजी, और समाप्ति अभी भी वर्तमान थे, ठीक एक बार। यदि किसी भी पक्ष में बदलाव होता है, तो चेन बंद हो जाती है, और अस्वीकार का कारण आपको बताता है कि **कौन सा** गुण टूटा: पुरानी अधिकारिता बनाम बदली गई क्रिया।

**साबित नहीं करता:** कि मंजूरी UI ने मानव को वही दिखाया जो वे साइन कर रहे थे (WYSIWYS अपनी खुद की समस्या है), कि कुंजी रोटेशन से पहले जबरदस्ती या चोरी नहीं हुई थी, या कि डाउनस्ट्रीम प्रभाव क्रिया से मेल खाते थे। हस्ताक्षरित ≠ अधिकृत: एक पुरानी नीति के ऊपर एक मान्य हस्ताक्षर, एक बदली हुई कुंजी, एक समाप्त विंडो, या एक अलग डाइजेस्ट यहाँ कुछ भी प्रमाणित नहीं करता।

दोनों रसीद प्रकार उद्देश्य से पाठ की लिफाफा और एक `verify_chain` कोड पथ साझा करते हैं: मुख्य नोटबुक में क्रिया रसीदों के लिए आपने जो बाइंडिंग बनाई है, वही कोड मानव की मंजूरी जांचता है। एक प्रमाणक अनुबंध, अलग-अलग पिन की गई अधिकारियां, कैनॉनिकल क्रिया डाइजेस्ट द्वारा संयुक्त और कुछ नहीं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
